In [ ]:
%pip install -U langchain langchain-community langchain-core langchain-databricks faiss-cpu sentence-transformers mlflow pypdf
dbutils.library.restartPython()

In [0]:
import os
import json
import pickle
import mlflow
from typing import List, Dict, Any, Tuple

from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_databricks import ChatDatabricks

In [0]:

CONFIG_PATH = "dbfs:/Volumes/main/default/dino_pdfs/project/config/project_config.json"
config = json.loads(dbutils.fs.head(CONFIG_PATH, 100000))

INDEX_URI = config["index_uri"]
MLFLOW_EXPERIMENT = config["mlflow_experiment"]
EMBEDDING_MODEL_NAME = config.get("embedding_model_name", "sentence-transformers/all-MiniLM-L6-v2")
CHAT_ENDPOINT = config.get("chat_endpoint", "databricks-meta-llama-3-1-8b-instruct")
TOP_K = int(config.get("top_k", 4))

mlflow.set_experiment(MLFLOW_EXPERIMENT)

print("Loaded config:")
print(json.dumps(config, indent=2))

Loaded config:
{
  "project_name": "DinoRAG",
  "domain_name": "Dinosaurs",
  "catalog": "main",
  "schema": "default",
  "volume": "dino_pdfs",
  "volume_uri": "/Volumes/main/default/dino_pdfs",
  "project_uri": "/Volumes/main/default/dino_pdfs/project",
  "artifacts_uri": "/Volumes/main/default/dino_pdfs/project/artifacts",
  "index_uri": "/Volumes/main/default/dino_pdfs/project/faiss_index",
  "config_uri": "/Volumes/main/default/dino_pdfs/project/config",
  "mlflow_experiment": "/Shared/Dinosaur_RAG_Experiments",
  "chunk_size": 800,
  "chunk_overlap": 150,
  "top_k": 4,
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "chat_endpoint": "databricks-meta-llama-3-1-8b-instruct",
  "delta_table": "main.default.dinosaur_chunks"
}


In [0]:

try:
    mlflow.langchain.autolog()
    print("MLflow LangChain autolog enabled.")
except Exception as e:
    print("MLflow LangChain autolog could not be enabled:", str(e))

MLflow LangChain autolog enabled.


In [0]:

class SBERTEmbeddings(Embeddings):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

In [0]:
local_index_dir = "/Volumes/main/default/dino_pdfs/project/temp_faiss_index_runtime"
os.makedirs(local_index_dir, exist_ok=True)

for item in dbutils.fs.ls(INDEX_URI):
    if not item.isDir():
        dbutils.fs.cp(item.path, f"{local_index_dir}/{item.name}", True)

embeddings = SBERTEmbeddings(EMBEDDING_MODEL_NAME)

vectorstore = FAISS.load_local(
    local_index_dir,
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print("FAISS index loaded successfully.")
print("Top-k:", TOP_K)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index loaded successfully.
Top-k: 4


In [0]:
# -----------------------------
# Databricks chat model
# Same model is used for both:
# 1) PDF-grounded answers
# 2) External knowledge fallback
# -----------------------------
llm = ChatDatabricks(
    endpoint=CHAT_ENDPOINT,
    temperature=0.1,
    max_tokens=500
)

print("Chat endpoint configured:", CHAT_ENDPOINT)

Chat endpoint configured: databricks-meta-llama-3-1-8b-instruct


In [0]:
# -----------------------------
# Memory
# -----------------------------
chat_history: List[Dict[str, str]] = []

def format_history(history: List[Dict[str, str]], max_turns: int = 3) -> str:
    recent = history[-max_turns:]
    if not recent:
        return "No previous conversation."
    return "\n".join(
        [f"User: {item['user']}\nAssistant: {item['assistant']}" for item in recent]
    )

In [0]:
# -----------------------------
# Helper functions for retrieval
# -----------------------------
def retrieve_with_scores(question: str, k: int = None) -> List[Tuple[Any, float]]:
    if k is None:
        k = TOP_K
    return vectorstore.similarity_search_with_score(question, k=k)

def build_context_from_docs(docs: List[Any]) -> str:
    return "\n\n".join([d.page_content for d in docs])

def has_strong_pdf_context(
    docs: List[Any],
    context: str,
    scores: List[float],
    min_docs: int = 2,
    min_total_chars: int = 300,
    max_avg_distance: float = 1.6
) -> bool:
    if not docs or len(docs) < min_docs:
        return False

    if len(context.strip()) < min_total_chars:
        return False

    if not scores:
        return False

    avg_distance = sum(scores) / len(scores)
    return avg_distance <= max_avg_distance

In [0]:
# -----------------------------
# Tool-like helper functions
# -----------------------------
def source_lookup_tool(question: str, k: int = None) -> List[Dict[str, Any]]:
    results = retrieve_with_scores(question, k=k)
    output = []
    for doc, score in results:
        output.append({
            "source_file": doc.metadata.get("source_file"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "distance_score": float(score),
            "preview": doc.page_content[:250]
        })
    return output

def period_lookup_tool(period_name: str) -> List[Dict[str, Any]]:
    safe_period = period_name.replace("'", " ")
    query = f'''
    SELECT source_file, page, chunk_id, text
    FROM main.default.dinosaur_chunks
    WHERE lower(text) LIKE '%{safe_period.lower()}%'
    LIMIT 5
    '''
    return [row.asDict() for row in spark.sql(query).collect()]

In [0]:
# -----------------------------
# Prompt templates
# -----------------------------
PDF_SYSTEM_PROMPT = (
    "You are a dinosaur domain assistant. "
    "Answer only from the provided PDF context. "
    "Do not use outside knowledge. "
    "If the answer is not supported by the context, say: "
    "'I could not find enough information in the PDF knowledge base.' "
    "Be concise, factual, and clear."
)

EXTERNAL_SYSTEM_PROMPT = (
    "You are a helpful assistant with general dinosaur knowledge. "
    "The internal dinosaur PDF knowledge base did not provide enough information. "
    "Answer the question using general knowledge only. "
    "Do not claim that the answer came from the PDFs. "
    "Be concise, factual, and clear."
)

In [0]:
# -----------------------------
# Response generators
# -----------------------------
def call_llm(system_prompt: str, user_prompt: str) -> str:
    messages = [
        ("system", system_prompt),
        ("human", user_prompt)
    ]
    response = llm.invoke(messages)
    return response.content

def answer_from_pdf(question: str, docs: List[Any]) -> Dict[str, Any]:
    context = build_context_from_docs(docs)

    user_prompt = f'''
Conversation history:
{format_history(chat_history)}

Context:
{context}

Question:
{question}

Answer:
'''

    answer = call_llm(PDF_SYSTEM_PROMPT, user_prompt)

    sources = [
        {
            "source_file": d.metadata.get("source_file"),
            "page": d.metadata.get("page"),
            "chunk_id": d.metadata.get("chunk_id")
        }
        for d in docs
    ]

    return {
        "mode": "pdf",
        "label": "Answer source: PDF Knowledge Base",
        "answer": answer,
        "sources": sources
    }

def answer_from_external(question: str) -> Dict[str, Any]:
    user_prompt = f'''
Conversation history:
{format_history(chat_history)}

Question:
{question}

Answer:
'''

    answer = call_llm(EXTERNAL_SYSTEM_PROMPT, user_prompt)

    return {
        "mode": "external",
        "label": "Answer source: External Knowledge",
        "answer": answer,
        "sources": []
    }

In [0]:
# -----------------------------
# Main routing function
# -----------------------------
def ask_dino_bot(question: str, k: int = None, save_memory: bool = True) -> Dict[str, Any]:
    if k is None:
        k = TOP_K

    results = retrieve_with_scores(question, k=k)
    docs = [doc for doc, score in results]
    scores = [float(score) for doc, score in results]
    context = build_context_from_docs(docs)

    use_pdf = has_strong_pdf_context(
        docs=docs,
        context=context,
        scores=scores,
        min_docs=2,
        min_total_chars=300,
        max_avg_distance=1.6
    )

    if use_pdf:
        result = answer_from_pdf(question, docs)
    else:
        result = answer_from_external(question)

    result["retrieval_scores"] = scores

    if save_memory:
        chat_history.append({
            "user": question,
            "assistant": result["answer"]
        })

    return result

In [0]:
# -----------------------------
# Quick tests
# -----------------------------
test_questions = [
    "Tell me about Triceratops.",
    "What did Stegosaurus eat?",
    "Which dinosaur lived on Mars?"
]

for q in test_questions:
    result = ask_dino_bot(q, save_memory=False)
    print("=" * 80)
    print("QUESTION:", q)
    print(result["label"])
    print(result["answer"][:800])
    print("Sources:", result["sources"][:3])
    print("Scores:", result["retrieval_scores"])

QUESTION: Tell me about Triceratops.
Answer source: PDF Knowledge Base
Triceratops is a genus of dinosaur that is included in the definition of Dinosauria as the most recent common ancestor (MRCA) of Triceratops and modern birds (Neornithes), and all its descendants.
Sources: [{'source_file': 'Document6_Dinosaur_Reproduction_Parental_Care_and_Migration Behavior.pdf', 'page': 1, 'chunk_id': 126}, {'source_file': 'Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'page': 1, 'chunk_id': 5}, {'source_file': 'Document1_Dinosaur_Classification_and_General_Characteristics.pdf', 'page': 9, 'chunk_id': 37}]
Scores: [1.172004222869873, 1.2316474914550781, 1.232895851135254, 1.2472107410430908]


QUESTION: What did Stegosaurus eat?
Answer source: PDF Knowledge Base
I could not find enough information in the PDF knowledge base about what Stegosaurus ate.
Sources: [{'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 4, 'chunk_id': 75}, {'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 3, 'chunk_id': 165}, {'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 1, 'chunk_id': 61}]
Scores: [0.7887638807296753, 0.8011966347694397, 0.9182360172271729, 0.923947811126709]


QUESTION: Which dinosaur lived on Mars?
Answer source: PDF Knowledge Base
I could not find enough information in the PDF knowledge base.
Sources: [{'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 2, 'chunk_id': 162}, {'source_file': 'Document3_Dinosaur_Classification_Size_and_Behavior.pdf', 'page': 1, 'chunk_id': 64}, {'source_file': 'Document8_Common_Misconceptions_About_Dinosaurs.pdf', 'page': 3, 'chunk_id': 166}]
Scores: [0.9336845874786377, 1.0095248222351074, 1.0096521377563477, 1.0150883197784424]


In [0]:
# -----------------------------
# Log one demo run to MLflow
# -----------------------------
with mlflow.start_run(run_name="03_rag_pipeline_with_memory_tools_improved"):
    sample_question = "Compare Triceratops and Tyrannosaurus rex."
    sample_result = ask_dino_bot(sample_question, save_memory=False)

    mlflow.log_param("chat_endpoint", CHAT_ENDPOINT)
    mlflow.log_param("embedding_model_name", EMBEDDING_MODEL_NAME)
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("fallback_strategy", "same_model_two_prompts")
    mlflow.log_text(json.dumps(sample_result, indent=2), "sample_rag_result.json")

print("Notebook 4 completed successfully.")

Notebook 4 completed successfully.
